# Heart Disease Prediction Project

Includes:
- Data Loading
- EDA
- Missing Value Analysis
- Missing Value Handling
- Feature Engineering
- pd.get_dummies(drop_first=True)
- Logistic Regression
- Random Forest
- Accuracy, Classification Report
- Confusion Matrix


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

pd.set_option('display.max_columns', None)

df = pd.read_csv('/content/heart_disease_uci(3).csv')

print("Shape:", df.shape)
display(df.head())


In [ ]:

# Basic EDA

print(df.info())
print("\nMissing Values\n")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

display(df.describe(include='all'))


In [ ]:

# Convert target to binary

df['num'] = (df['num'] > 0).astype(int)

print(df['num'].value_counts())


In [ ]:

# Missing Value Handling

numeric_cols = ['trestbps','chol','thalch','oldpeak']

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in ['fbs','restecg','exang']:
    df[col] = df[col].fillna(df[col].mode()[0])

df['slope'] = df['slope'].fillna('Missing')
df['thal'] = df['thal'].fillna('Missing')

# ca has many missing values
df['ca'] = df['ca'].fillna(-1)

print(df.isnull().sum())


In [ ]:

# Feature Engineering

df['ca_missing'] = (df['ca'] == -1).astype(int)
df['thal_missing'] = (df['thal'] == 'Missing').astype(int)

df['age_oldpeak'] = df['age'] * df['oldpeak']


In [ ]:

# Prepare Features

X = df.drop(columns=['num'])

if 'id' in X.columns:
    X = X.drop(columns=['id'])

y = df['num']

X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

print("Encoded Shape:", X.shape)
display(X.head())


In [ ]:

# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=59,
    stratify=y
)

print(X_train.shape, X_test.shape)


In [ ]:

# Logistic Regression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(
    max_iter=3000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))

print(classification_report(y_test, lr_pred))

cm = confusion_matrix(y_test, lr_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Logistic Regression Confusion Matrix')
plt.show()


In [ ]:

# Random Forest

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

print(classification_report(y_test, rf_pred))

cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Random Forest Confusion Matrix')
plt.show()
